In [1]:
import time
t0 = time.time()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("phase2-csv-to-parquet").getOrCreate()
print(f"session up in {time.time()-t0:.1f}s")

session up in 9.6s


In [2]:
DATA_DIR = "/home/jovyan/work/data"
OUT_DIR  = "/home/jovyan/work/data/processed"

tx = spark.read.csv(f"{DATA_DIR}/train_transaction.csv", header=True, inferSchema=True)
t0 = time.time()
tx.write.mode("overwrite").parquet(f"{OUT_DIR}/train_transaction")
print(f"train_transaction written in {time.time()-t0:.1f}s")

tx_test = spark.read.csv(f"{DATA_DIR}/test_transaction.csv", header=True, inferSchema=True)
t0 = time.time()
tx_test.write.mode("overwrite").parquet(f"{OUT_DIR}/test_transaction")
print(f"test_transaction written in {time.time()-t0:.1f}s")

train_transaction written in 76.4s
test_transaction written in 50.4s


In [3]:
idn = spark.read.csv(f"{DATA_DIR}/train_identity.csv", header=True, inferSchema=True)
idn.write.mode("overwrite").parquet(f"{OUT_DIR}/train_identity")

idn_test = spark.read.csv(f"{DATA_DIR}/test_identity.csv", header=True, inferSchema=True)
idn_test.write.mode("overwrite").parquet(f"{OUT_DIR}/test_identity")
print("identity files written")

identity files written


In [5]:
expected = {"train_transaction": 590540, "test_transaction": 506691,
            "train_identity": 144233, "test_identity": 141907}
for name in expected:
    n = spark.read.parquet(f"{OUT_DIR}/{name}").count()
    ok = "✅" if n == expected[name] else "❌"
    print(f"{ok} {name}: {n:,} (expected {expected[name]:,})")

✅ train_transaction: 590,540 (expected 590,540)
✅ test_transaction: 506,691 (expected 506,691)
✅ train_identity: 144,233 (expected 144,233)
✅ test_identity: 141,907 (expected 141,907)


In [6]:
csv_df = spark.read.csv(f"{DATA_DIR}/train_transaction.csv", header=True, inferSchema=True)
pq_df  = spark.read.parquet(f"{OUT_DIR}/train_transaction")

t0 = time.time(); csv_df.select("isFraud").groupBy("isFraud").count().collect(); csv_t = time.time()-t0
t0 = time.time(); pq_df.select("isFraud").groupBy("isFraud").count().collect();  pqt_t  = time.time()-t0
print(f"groupBy on CSV : {csv_t:.1f}s   |   groupBy on Parquet: {pqt_t:.1f}s   |   speedup: {csv_t/pqt_t:.1f}x")

groupBy on CSV : 7.7s   |   groupBy on Parquet: 1.2s   |   speedup: 6.3x
